# Cross-Pollination Lab — Pangea-Earth

**Mycelium applied to art. Each sprite is a node. Cross-pollination is the contract.**

Pipeline:
1. Extract — clean isolation per sprite (Sierpiński L0→L1→L2)
2. Decompose — head/torso/arms/legs as modular slots
3. Cross-pollinate — swap parts between characters (contracts between nodes)
4. Fill blanks — generate missing angles/poses via depth estimation
5. Skin system — melanin ratio as transferable color palette

Same sieve architecture: decompose, filter, recombine.

Guinea Pig Trench LLC

In [ ]:
#@title 1. Setup — Mount Drive, load all sprites
import os, time, json, shutil, io
import numpy as np
import cv2
from PIL import Image
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

!pip install -q rembg onnxruntime-gpu

DRIVE_BASE = Path('/content/drive/MyDrive/Guinea Pig Trench')
SOURCE_DIR = DRIVE_BASE / 'sprites' / 'source'
ENHANCED_DIR = DRIVE_BASE / 'game-assets' / 'enhanced'
GENERATED_DIR = DRIVE_BASE / 'game-assets' / 'generated'
OUTPUT_DIR = DRIVE_BASE / 'sprites' / 'cross_pollinated'
PARTS_DIR = DRIVE_BASE / 'sprites' / 'parts'
SKINS_DIR = DRIVE_BASE / 'sprites' / 'skins'

for d in [OUTPUT_DIR, PARTS_DIR, SKINS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SESSION_START = time.time()
SESSION_LIMIT = 80 * 60
def time_left(): return max(0, SESSION_LIMIT - (time.time() - SESSION_START))

# Load all source sprites
sources = sorted(SOURCE_DIR.glob('*.png'))
enhanced = sorted(ENHANCED_DIR.glob('*.png')) if ENHANCED_DIR.exists() else []
generated = sorted(GENERATED_DIR.rglob('*.png')) if GENERATED_DIR.exists() else []

print(f'Source sprites: {len(sources)}')
for s in sources:
    img = Image.open(s)
    print(f'  {s.name}: {img.size[0]}x{img.size[1]}')
print(f'Enhanced: {len(enhanced)}')
print(f'Generated (multi-angle): {len(generated)}')
print(f'\nSession budget: {SESSION_LIMIT//60} min')

In [ ]:
#@title 2. Clean Extraction — background removal + frame splitting
from rembg import remove as rembg_remove

def clean_extract(img):
    """L0→L1→L2 Sierpinski cleanup. Cheap first."""
    # L1: Find subject bbox at thumbnail
    w, h = img.size
    thumb = img.convert('RGBA').resize((64, 64), Image.NEAREST)
    arr = np.array(thumb)
    if arr.shape[2] == 4:
        mask = arr[:,:,3] > 20
    else:
        gray = np.mean(arr[:,:,:3], axis=2)
        mask = np.abs(gray - np.median(gray)) > 30
    
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if rows.any() and cols.any():
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
        pad = 2
        sx, sy = w/64, h/64
        bbox = (max(0,int((cmin-pad)*sx)), max(0,int((rmin-pad)*sy)),
                min(w,int((cmax+pad+1)*sx)), min(h,int((rmax+pad+1)*sy)))
        cropped = img.crop(bbox)
    else:
        cropped = img
    
    # L2: rembg on cropped region
    buf = io.BytesIO()
    cropped.save(buf, format='PNG')
    result = rembg_remove(buf.getvalue())
    return Image.open(io.BytesIO(result)).convert('RGBA')


def split_frames(img, name, min_area_pct=0.005):
    """Split sprite sheet into individual frames via connected components."""
    arr = np.array(img.convert('RGBA'))
    h, w = arr.shape[:2]
    
    # Coarse detection at 1/4 res
    small = cv2.resize(arr, (w//4, h//4), interpolation=cv2.INTER_NEAREST)
    if small.shape[2] == 4:
        mask = (small[:,:,3] > 20).astype(np.uint8) * 255
    else:
        gray = cv2.cvtColor(small[:,:,:3], cv2.COLOR_RGB2GRAY)
        mask = (np.abs(gray.astype(float) - np.median(gray)) > 30).astype(np.uint8) * 255
    
    kernel = np.ones((3,3), np.uint8)
    dilated = cv2.dilate(mask, kernel, iterations=2)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dilated)
    
    frames = []
    min_area = (w//4 * h//4) * min_area_pct
    for i in range(1, n_labels):
        x, y, fw, fh, area = stats[i]
        if area < min_area:
            continue
        pad = 8
        x1 = max(0, x*4 - pad)
        y1 = max(0, y*4 - pad)
        x2 = min(w, (x+fw)*4 + pad)
        y2 = min(h, (y+fh)*4 + pad)
        frames.append(img.crop((x1, y1, x2, y2)))
    
    return frames


# Process all source sprites
print('=== Clean Extraction ===')
cleaned_sprites = {}

for src in sources:
    if time_left() < 300:
        print(f'  TIME — stopping')
        break
    
    img = Image.open(src)
    name = src.stem
    print(f'  {src.name} ({img.size[0]}x{img.size[1]}):', end=' ')
    
    # Check if it's a sprite sheet (multiple subjects)
    frames = split_frames(img, name)
    
    if len(frames) > 1:
        print(f'{len(frames)} frames detected')
        cleaned = []
        for j, frame in enumerate(frames):
            clean = clean_extract(frame)
            out_path = PARTS_DIR / f'{name}_frame_{j:02d}.png'
            clean.save(out_path)
            cleaned.append(clean)
        cleaned_sprites[name] = cleaned
    else:
        clean = clean_extract(img)
        out_path = PARTS_DIR / f'{name}_clean.png'
        clean.save(out_path)
        cleaned_sprites[name] = [clean]
        print('single subject')

print(f'\n{sum(len(v) for v in cleaned_sprites.values())} total cleaned frames')
print(f'Saved to {PARTS_DIR}')

In [ ]:
#@title 3. Decompose — extract body part regions per character

def decompose_character(img, name):
    """Split a character sprite into approximate body regions.
    Uses vertical/horizontal slicing based on proportions.
    Head = top 30%, Torso = 30-55%, Arms = sides of torso, Legs = bottom 45%."""
    
    arr = np.array(img.convert('RGBA'))
    h, w = arr.shape[:2]
    
    # Find actual content bounds (non-transparent)
    if arr.shape[2] == 4:
        alpha = arr[:,:,3]
        rows = np.any(alpha > 20, axis=1)
        cols = np.any(alpha > 20, axis=0)
        if not rows.any():
            return {}
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]
    else:
        rmin, rmax, cmin, cmax = 0, h-1, 0, w-1
    
    ch = rmax - rmin
    cw = cmax - cmin
    
    parts = {}
    
    # Head: top 30%
    head_bottom = rmin + int(ch * 0.30)
    parts['head'] = img.crop((cmin, rmin, cmax, head_bottom))
    
    # Torso: 30-55%
    torso_top = head_bottom
    torso_bottom = rmin + int(ch * 0.55)
    center_left = cmin + int(cw * 0.2)
    center_right = cmin + int(cw * 0.8)
    parts['torso'] = img.crop((center_left, torso_top, center_right, torso_bottom))
    
    # Left arm: left side of torso region
    parts['arm_left'] = img.crop((cmin, torso_top, center_left, torso_bottom))
    
    # Right arm: right side of torso region
    parts['arm_right'] = img.crop((center_right, torso_top, cmax, torso_bottom))
    
    # Legs: bottom 45%
    legs_top = torso_bottom
    parts['legs'] = img.crop((cmin, legs_top, cmax, rmax))
    
    return parts


print('=== Body Part Decomposition ===')
character_parts = {}

# Only decompose single-subject characters (not sprite sheets with many frames)
single_chars = ['aku_aku_mask_stylized', 'mecha_entity_alpha_v2', 
                'kraken_game_render', 'void_runner_ship',
                'mecha_entity_alpha_v2_pixel']

for name in single_chars:
    clean_path = PARTS_DIR / f'{name}_clean.png'
    if not clean_path.exists():
        # Try using source directly
        src_path = SOURCE_DIR / f'{name}.png'
        if src_path.exists():
            clean_path = src_path
        else:
            continue
    
    img = Image.open(clean_path)
    parts = decompose_character(img, name)
    
    if parts:
        character_parts[name] = parts
        print(f'  {name}: {len(parts)} parts')
        for part_name, part_img in parts.items():
            part_path = PARTS_DIR / f'{name}_{part_name}.png'
            part_img.save(part_path)
            print(f'    {part_name}: {part_img.size[0]}x{part_img.size[1]}')

# Also decompose first frame of multi-frame sprites
for name, frames in cleaned_sprites.items():
    if name in single_chars or len(frames) == 0:
        continue
    parts = decompose_character(frames[0], name)
    if parts:
        character_parts[name] = parts
        print(f'  {name} (frame 0): {len(parts)} parts')
        for part_name, part_img in parts.items():
            part_path = PARTS_DIR / f'{name}_{part_name}.png'
            part_img.save(part_path)

print(f'\n{len(character_parts)} characters decomposed')
print(f'Saved to {PARTS_DIR}')

In [ ]:
#@title 4. Cross-Pollinate — swap parts between characters

def cross_pollinate(base_name, donor_name, part_to_swap, character_parts):
    """Take a body part from donor and put it on base character.
    The mycelium contract: two nodes exchange resources."""
    
    if base_name not in character_parts or donor_name not in character_parts:
        return None
    
    base_parts = character_parts[base_name]
    donor_parts = character_parts[donor_name]
    
    if part_to_swap not in base_parts or part_to_swap not in donor_parts:
        return None
    
    # Get base character full image
    base_clean = PARTS_DIR / f'{base_name}_clean.png'
    if not base_clean.exists():
        base_clean = SOURCE_DIR / f'{base_name}.png'
    base_img = Image.open(base_clean).convert('RGBA')
    
    # Get donor part, resize to match base part dimensions
    base_part = base_parts[part_to_swap]
    donor_part = donor_parts[part_to_swap]
    donor_resized = donor_part.resize(base_part.size, Image.LANCZOS)
    
    # Find where the part sits on the base image
    arr = np.array(base_img)
    h, w = arr.shape[:2]
    alpha = arr[:,:,3] if arr.shape[2] == 4 else np.ones((h,w), dtype=np.uint8) * 255
    rows = np.any(alpha > 20, axis=1)
    cols = np.any(alpha > 20, axis=0)
    if not rows.any():
        return None
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    ch, cw = rmax - rmin, cmax - cmin
    
    # Part position mapping
    positions = {
        'head': (cmin, rmin),
        'torso': (cmin + int(cw*0.2), rmin + int(ch*0.30)),
        'arm_left': (cmin, rmin + int(ch*0.30)),
        'arm_right': (cmin + int(cw*0.8), rmin + int(ch*0.30)),
        'legs': (cmin, rmin + int(ch*0.55)),
    }
    
    pos = positions.get(part_to_swap, (0, 0))
    result = base_img.copy()
    result.paste(donor_resized, pos, donor_resized)
    
    return result


print('=== Cross-Pollination ===')
print('Swapping parts between characters...')
print()

# Generate all possible crosses between available characters
char_names = list(character_parts.keys())
crosses = []

for base in char_names:
    for donor in char_names:
        if base == donor:
            continue
        for part in ['head', 'torso', 'legs']:
            if time_left() < 120:
                break
            result = cross_pollinate(base, donor, part, character_parts)
            if result:
                cross_name = f'{base}_x_{donor}_{part}'
                out_path = OUTPUT_DIR / f'{cross_name}.png'
                result.save(out_path)
                crosses.append(cross_name)
                print(f'  {base} + {donor} ({part}) → {cross_name}')

print(f'\n{len(crosses)} cross-pollinated sprites generated')
print(f'Saved to {OUTPUT_DIR}')

In [ ]:
#@title 5. Skin System — extract and transfer color palettes (melanin ratio)

def extract_palette(img, n_colors=5):
    """Extract dominant color palette from sprite.
    Returns sorted by luminance (darkest to brightest).
    Each color is a point on the eumelanin/pheomelanin spectrum."""
    arr = np.array(img.convert('RGBA'))
    
    # Only sample non-transparent pixels
    if arr.shape[2] == 4:
        mask = arr[:,:,3] > 20
        pixels = arr[mask][:,:3]
    else:
        pixels = arr.reshape(-1, 3)
    
    if len(pixels) < n_colors:
        return []
    
    # K-means clustering for dominant colors
    from sklearn.cluster import MiniBatchKMeans
    kmeans = MiniBatchKMeans(n_clusters=n_colors, random_state=42, n_init=3)
    kmeans.fit(pixels.astype(float))
    colors = kmeans.cluster_centers_.astype(int)
    
    # Sort by luminance
    lum = colors[:,0] * 0.299 + colors[:,1] * 0.587 + colors[:,2] * 0.114
    order = np.argsort(lum)
    colors = colors[order]
    
    return colors.tolist()


def apply_skin(img, source_palette, target_palette):
    """Remap colors from source palette to target palette.
    Same structure, different melanin ratio."""
    arr = np.array(img.convert('RGBA')).astype(float)
    rgb = arr[:,:,:3]
    alpha = arr[:,:,3:]
    
    source = np.array(source_palette, dtype=float)
    target = np.array(target_palette, dtype=float)
    
    if len(source) != len(target) or len(source) == 0:
        return img
    
    # For each pixel, find closest source color and blend toward target
    result = rgb.copy()
    for i in range(len(source)):
        dist = np.sqrt(np.sum((rgb - source[i])**2, axis=2))
        weight = np.exp(-dist / 50.0)  # Gaussian weight
        shift = target[i] - source[i]
        for c in range(3):
            result[:,:,c] += weight * shift[c]
    
    result = np.clip(result, 0, 255)
    output = np.concatenate([result, alpha], axis=2).astype(np.uint8)
    return Image.fromarray(output)


print('=== Skin System — Color Palette Extraction ===')
print()

!pip install -q scikit-learn
from sklearn.cluster import MiniBatchKMeans

palettes = {}
import matplotlib.pyplot as plt

fig_rows = min(len(sources), 9)
fig, axes = plt.subplots(fig_rows, 2, figsize=(12, 2*fig_rows))
if fig_rows == 1:
    axes = [axes]

for i, src in enumerate(sources[:fig_rows]):
    img = Image.open(src)
    palette = extract_palette(img, n_colors=5)
    palettes[src.stem] = palette
    
    # Show sprite + palette
    axes[i][0].imshow(np.array(img))
    axes[i][0].set_title(src.stem[:25], fontsize=8)
    axes[i][0].axis('off')
    
    if palette:
        swatch = np.array(palette, dtype=np.uint8).reshape(1, -1, 3)
        swatch = np.repeat(swatch, 30, axis=0)
        axes[i][1].imshow(swatch)
        axes[i][1].set_title('palette (dark→light)', fontsize=8)
    axes[i][1].axis('off')

plt.tight_layout()
plt.savefig(str(SKINS_DIR / 'all_palettes.png'), dpi=150)
plt.show()

# Cross-apply skins
print('\n=== Skin Transfer ===')
skin_count = 0
char_names_with_palettes = [k for k in palettes if len(palettes[k]) == 5]

for base_name in char_names_with_palettes[:4]:
    for donor_name in char_names_with_palettes[:4]:
        if base_name == donor_name:
            continue
        if time_left() < 60:
            break
        
        base_path = PARTS_DIR / f'{base_name}_clean.png'
        if not base_path.exists():
            base_path = SOURCE_DIR / f'{base_name}.png'
        if not base_path.exists():
            continue
        
        base_img = Image.open(base_path)
        skinned = apply_skin(base_img, palettes[base_name], palettes[donor_name])
        out_path = SKINS_DIR / f'{base_name}_skin_{donor_name}.png'
        skinned.save(out_path)
        skin_count += 1
        print(f'  {base_name} → {donor_name} palette')

print(f'\n{skin_count} skin variants generated')
(SKINS_DIR / 'palettes.json').write_text(json.dumps({k: v for k, v in palettes.items()}, indent=2))
print(f'Palettes saved to {SKINS_DIR / "palettes.json"}')

In [ ]:
#@title 6. Session Summary

results = {
    'session_date': time.strftime('%Y-%m-%d %H:%M'),
    'sprites_cleaned': sum(len(v) for v in cleaned_sprites.values()) if 'cleaned_sprites' in dir() else 0,
    'characters_decomposed': len(character_parts) if 'character_parts' in dir() else 0,
    'crosses_generated': len(crosses) if 'crosses' in dir() else 0,
    'skins_generated': skin_count if 'skin_count' in dir() else 0,
    'palettes_extracted': len(palettes) if 'palettes' in dir() else 0,
    'duration_min': round((time.time() - SESSION_START) / 60, 1),
}

(OUTPUT_DIR / 'cross_pollination_session.json').write_text(json.dumps(results, indent=2))

print('=== Cross-Pollination Lab — Session Summary ===')
print(f'Duration: {results["duration_min"]} min')
print(f'Sprites cleaned: {results["sprites_cleaned"]}')
print(f'Characters decomposed: {results["characters_decomposed"]}')
print(f'Cross-pollinated variants: {results["crosses_generated"]}')
print(f'Skin transfers: {results["skins_generated"]}')
print(f'Palettes extracted: {results["palettes_extracted"]}')
print(f'Time remaining: {time_left()/60:.0f} min')
print()
print('Each sprite is a node. Each swap is a contract.')
print('The mycelium grows between characters.')
print('Pangea-Earth. Guinea Pig Trench LLC')